# Case 01 — Retention Intelligence Platform  

### Apple Decision Intelligence Portfolio  
**Author:** Israel Gómez Millán  

---

## Executive Summary

This case presents an end-to-end retention intelligence framework designed to identify subscription churn risk, explain the behavioral causes of disengagement, and operationalize predictive outputs into actionable retention playbooks.

The project simulates a production-style machine learning workflow combining:

- Behavioral Simulation  
- Feature Engineering  
- Predictive Modeling  
- SHAP Explainability  
- Decision Intelligence  

---

### Business Goal

Improve subscription retention by proactively identifying at-risk users before churn occurs and prioritizing interventions by revenue impact.


## 1. Business Problem

Subscription platforms face continuous retention pressure as user engagement fluctuates over time.

The business challenge is not only to predict which users are likely to churn, but also to answer:

- Why are they at risk?
- Which users should be prioritized first?
- What intervention should be recommended?

Without this layer of intelligence, retention strategies become reactive, generic, and inefficient.


## 2. Analytical Objective

This project was designed to answer four core questions:

1. Which users are likely to churn within the next 30 days?
2. Which behavioral signals are most predictive of churn?
3. How should users be prioritized based on both risk and value?
4. Which type of retention action is most appropriate for each user?

The end result is not just a churn model, but a retention intelligence system.


## 3. Solution Architecture

The project was implemented as a modular pipeline:

**Simulation → Features → Modeling → Explainability → Decisioning**

This architecture mirrors production-style analytical workflows where model outputs must ultimately support business decisions, not just technical evaluation.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path("..").resolve()

REPORTS_DIR = PROJECT_ROOT / "reports"
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"

model_benchmark_path = REPORTS_DIR / "model_benchmark_results.csv"
shap_global_path = REPORTS_DIR / "shap_global_importance.csv"
decision_table_path = REPORTS_DIR / "decision_table.csv"
decision_tier_summary_path = REPORTS_DIR / "decision_tier_summary.csv"
decision_playbook_summary_path = REPORTS_DIR / "decision_playbook_summary.csv"
feature_table_path = DATA_DIR / "features" / "feature_table.csv"
simulation_master_path = DATA_DIR / "raw" / "simulation_master_table.csv"


In [ ]:
benchmark_df = pd.read_csv(model_benchmark_path)
shap_global_df = pd.read_csv(shap_global_path)
decision_table_df = pd.read_csv(decision_table_path)
decision_tier_summary_df = pd.read_csv(decision_tier_summary_path)
decision_playbook_summary_df = pd.read_csv(decision_playbook_summary_path)
feature_table_df = pd.read_csv(feature_table_path)
simulation_master_df = pd.read_csv(simulation_master_path)

print("Artifacts loaded successfully.")
print(f"Benchmark rows: {len(benchmark_df):,}")
print(f"Decision rows: {len(decision_table_df):,}")
print(f"Feature table rows: {len(feature_table_df):,}")
print(f"Simulation rows: {len(simulation_master_df):,}")


## 4. Simulation Review

The simulation layer generates a synthetic subscription ecosystem with:

- user profile metadata,
- engagement behavior,
- content consumption patterns,
- platform adoption metrics,
- probabilistically calibrated churn labels.

This allows the case to demonstrate end-to-end modeling without relying on proprietary production data.


In [ ]:
simulation_master_df.head()

In [ ]:
simulation_master_df[
    [
        "sessions_last_30d",
        "watch_time_last_30d",
        "completion_rate",
        "skip_rate",
        "days_since_last_session",
        "will_churn_30d",
    ]
].describe().T

In [ ]:
churn_rate = simulation_master_df["will_churn_30d"].mean()
print(f"Observed churn rate: {churn_rate:.2%}")

### Simulation Assessment

The synthetic environment was designed to produce a realistic churn scenario with:

- non-trivial class balance,
- plausible overlap between retained and churned users,
- behavioral rather than demographic signal dominance.

This is important because unrealistic simulation can artificially inflate downstream model performance.


## 5. Feature Engineering Review

The feature engineering layer transformed raw user behavior into a cleaner, leakage-aware modeling table.

Feature families included:

- engagement frequency,
- content consumption intensity,
- quality indicators,
- discovery metrics,
- recency features,
- derived behavioral ratios.


In [ ]:
feature_table_df.head()

In [ ]:
feature_table_df.shape

In [ ]:
feature_table_df.columns.tolist()

### Feature Engineering Design Choice

A key design principle was to keep the feature table interpretable and business-valid.

Columns such as:

- `user_id`
- `behavioral_segment`
- `churn_probability`
- `churn_date`

were intentionally excluded from model fitting or treated as analysis-only fields to avoid shortcut learning and leakage risk.


## 6. Modeling Benchmark

Three models were benchmarked:

- Logistic Regression
- Random Forest
- XGBoost

Champion selection prioritized not only predictive quality, but also operational usefulness through metrics such as:

- ROC-AUC
- PR-AUC
- Lift@10%


In [ ]:
benchmark_df

In [ ]:
benchmark_df.sort_values(["split", "roc_auc"], ascending=[True, False])

In [ ]:
validation_df = benchmark_df[benchmark_df["split"] == "validation"].copy()
validation_df["selection_score"] = (
    0.40 * validation_df["roc_auc"]
    + 0.25 * validation_df["pr_auc"]
    + 0.20 * validation_df["lift_at_10pct"]
    + 0.15 * validation_df["f1"]
)
validation_df.sort_values("selection_score", ascending=False)

### Benchmark Interpretation

Among benchmarked candidate models, XGBoost delivered the strongest balance between ranking quality, PR performance, and top-decile lift.

This makes it the preferred production model because retention intervention capacity is typically constrained and prioritization efficiency is critical.


## 7. Explainability Review

To move from prediction to decision intelligence, SHAP was applied to the champion XGBoost model.

The explainability objective was to answer:

- Which features globally drive churn?
- Which behavioral signals dominate risk?
- Why is an individual user flagged as high risk?


In [ ]:
shap_global_df.head(10)

In [ ]:
shap_global_df.head(10)[["feature_name", "mean_abs_shap"]]

### Key Explainability Findings

Global SHAP analysis showed that churn is primarily driven by:

- declining recent watch time,
- lower completion rate,
- higher inactivity recency,
- fewer recent sessions,
- weaker content diversity.

This is strategically important because it shows churn is behavior-driven rather than demographic-driven.


In [ ]:
summary_plot_path = ASSETS_DIR / "shap_summary_plot.png"
bar_plot_path = ASSETS_DIR / "shap_bar_importance.png"

if summary_plot_path.exists():
    display(Image(filename=str(summary_plot_path)))

if bar_plot_path.exists():
    display(Image(filename=str(bar_plot_path)))

### Behavioral Insight

The strongest churn predictors indicate that churn is driven primarily by deteriorating engagement patterns rather than demographic characteristics.

Most influential signals include:

- declining watch time,
- reduced completion rate,
- inactivity recency,
- lower content diversity.

This suggests that behavioral disengagement monitoring should be the primary retention trigger.


## 8. Decisioning Outputs

The final layer operationalizes model outputs by combining:

- churn probability,
- SHAP-based driver signals,
- revenue exposure,
- and playbook recommendation logic.

This transforms the system from predictive analytics into actionable decision support.


In [ ]:
decision_tier_summary_df

In [ ]:
decision_playbook_summary_df

In [ ]:
decision_table_df.head(10)

### Decision Intelligence Interpretation

The final scoring layer transforms raw churn predictions into operational business decisions.

Users are segmented into actionable risk tiers and paired with personalized intervention playbooks based on:

- predicted churn probability,
- behavioral driver analysis,
- revenue exposure,
- and retention priority score.


In [ ]:
total_revenue_exposure = decision_table_df["revenue_at_risk"].sum()
critical_users = (decision_table_df["risk_tier"].str.lower() == "critical").sum()

print(f"Critical users identified: {critical_users:,}")
print(f"Total revenue exposure detected: ${total_revenue_exposure:,.0f}")

## 9. Strategic Recommendations

Based on the modeled outputs, three business recommendations emerge:

### 1. Prioritize critical high-value users first
A relatively small subset of users drives disproportionate revenue exposure.

### 2. Align intervention strategy with behavioral deterioration type
Different churn drivers require different playbooks.

### 3. Use engagement decay as the earliest warning signal
Recent consumption decline and inactivity are the strongest leading indicators of churn.


## 10. Final Portfolio Takeaways

This case demonstrates the ability to architect and productionize a complete decision intelligence workflow, including:

- predictive analytics,
- explainable machine learning,
- business prioritization frameworks,
- and operational retention strategy design.

Rather than stopping at prediction, the solution bridges analytical outputs into business execution—mirroring real-world Decision Intelligence team workflows.


## Appendix — Portfolio Positioning

This case is designed to demonstrate capability across:

- machine learning system design,
- behavioral analytics,
- explainable AI,
- business-oriented prioritization,
- and executive storytelling.

It is intended as a flagship portfolio case for roles in Apple Decision Intelligence and similar strategic analytics organizations.
